## Structured Output

### Structured Output means asking an LLM to return its answer in a predefined format instead of returning ordinary, unstructured text.
###      {
    "job_title": "AI/ML Engineer",
    "required_skills": ["Python", "Machine Learning", "SQL"],
    "experience_level": "Entry-level",
    "is_suitable_for_fresher": true
}

### The schema can be defined using:
1. Pydantic — best when you want validation.
2. TypedDict — useful for defining dictionary-shaped results.
3. JSON Schema — useful when you want to specify the schema directly.

## Pydantic

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# Initialize Groq model
model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.7,
)

# Generate response
response = model.invoke("What is API key?")

print(response.content)

### In a nutshell  
An **API key** is a short, unique string of characters that identifies the application (or user) making a request to an API. Think of it as a “password” that the API uses to:

1. **Authenticate** you – prove that you’re allowed to talk to the service.  
2. **Authorize** you – tell the service which actions you’re permitted to perform.  
3. **Track usage** – help the provider monitor how many calls you make, enforce quotas, and bill you if needed.

---

## How it works

| Step | What happens | What you see |
|------|--------------|--------------|
| 1. **Request** | Your app sends an HTTP request to the API endpoint. | `GET https://api.example.com/v1/users` |
| 2. **Key in header** | You include the key, usually in an HTTP header or query parameter. | `Authorization: Bearer abc123xyz` |
| 3. **API verifies** | The API server checks the key against its database. | Valid → proceed; invalid → 401 Unauthorized |
| 4. **Response** | The API returns data or an error. | `200

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the Movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000257DEF1A660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000257DEF1B380>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [4]:
response = model_with_structure.invoke("Provide details about the movie Increption")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [5]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with detailed """
    tilte:str = Field(..., description="The tile of the movie")
    year:int = Field(..., description="The year of the movie was released")
    director:str= Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movie's rating out of 10")

model_with_strucutre = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Increption")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Nested Structure

In [ ]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role: str


class MovieDetailed(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]

    budget: float | None = Field(
        description=(
            "Production budget in millions of USD. "
            "Return None if the budget is unknown."
        )
    )


model_with_structure = model.with_structured_output(MovieDetailed)

response = model_with_structure.invoke(
    """
    Provide detailed information about the movie Inception (2010).
    Include its production budget in millions of USD.
    If the budget is unknown, return None explicitly.
    """
)

print(response)
print("\nBudget:", response.budget)
print("All fields:", response.model_dump())

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')] genres=['Action', 'Adventure', 'Science Fiction', 'Thriller'] budget=160.0

Budget: 160.0
All fields: {'title': 'Inception', 'year': 2010, 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'}, {'name': 'Elliot Page', 'role': 'Ariadne'}, {'name': 'Tom Hardy', 'role': 'Eames'}, {'name': 'Ken Watanabe', 'role': 'Saito'}, {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}, {'name': 'Marion Cotillard', 'role': 'Mal'}, {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'}], 'genres': ['Action', 'Adventure', 'Science 

## TypedDict
### TypedDict is a Python feature used to define the expected keys and value types of a dictionary. In LangChain, you can use it as a schema for structured LLM output.

In [10]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str, ..., "The tile of the movie"]
    year:Annotated[int, ..., "The year of the movie was released"]
    director:Annotated[str, ..., "The director of the movie"]
    rating:Annotated[float, ..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2008}

In [13]:

from typing_extensions import TypedDict, Annotated


class Actor(TypedDict):
    name: str
    role: str


class MovieDetailed(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]

    budget: Annotated[
        float | None,
        "Production budget in millions of USD. "
        "Return None if the budget is unknown."
    ]


model_with_structure = model.with_structured_output(MovieDetailed)

response = model_with_structure.invoke(
    """
    Provide detailed information about Marvel's The Avengers (2012).
    Include the title, release year, cast with actor names and roles,
    genres, and production budget in millions of USD.
    If the budget is unknown, return None explicitly.
    """
)

print(response)
print("\nBudget:", response["budget"])
print("All fields:", response)


{'budget': 220, 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'}, {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}, {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'}, {'name': 'Chris Hemsworth', 'role': 'Thor'}, {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'}, {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}, {'name': 'Tom Hiddleston', 'role': 'Loki'}, {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}, {'name': 'Clark Gregg', 'role': 'Agent Coulson'}, {'name': 'Paul Bettany', 'role': 'J.A.R.V.I.S.'}], 'genres': ['Action', 'Adventure', 'Science Fiction', 'Fantasy', 'Superhero'], 'title': 'The Avengers', 'year': 2012}

Budget: 220
All fields: {'budget': 220, 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'}, {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}, {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'}, {'name': 'Chris Hemsworth', 'role': 'Thor'}, {'name': '

In [14]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data Classes

### n Python, a dataclass is a convenient way to define a class that stores structured data without writing a lot of repetitive code.
### In LangChain, you can use a dataclass to describe the structure of the information you want an LLM to return.
### Since you've learned Pydantic and TypedDict, let's compare them with dataclasses using your movie example.

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

class ContactInfo(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            Extract the contact information:
            Name: Ankur Kumar
            Email: ankur@example.com
            Phone: 9852569331
            """
        }
    ]
})

contact = response["structured_response"]

print(contact)
print("Name:", contact.name)
print("Email:", contact.email)
print("Phone:", contact.phone)

name='Ankur Kumar' email='ankur@example.com' phone='9852569331'
Name: Ankur Kumar
Email: ankur@example.com
Phone: 9852569331


In [17]:
## dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information for a person"""
    name:str
    email:str
    phone:int


model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            Extract the contact information:
            Name: Ankur Kumar
            Email: ankur@example.com
            Phone: 9852569331
            """
        }
    ]
})

contact = response["structured_response"]

print(contact)
print("Name:", contact.name)
print("Email:", contact.email)
print("Phone:", contact.phone)

ContactInfo(name='Ankur Kumar', email='ankur@example.com', phone=9852569331)
Name: Ankur Kumar
Email: ankur@example.com
Phone: 9852569331
